<a href="https://colab.research.google.com/github/vkantimahanti/healthcare-ml-portfolio/blob/main/sklearn_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install mlflow shap -q

'''
mlflow - Installs MLflow, an open-source platform used to manage the machine learning lifecycle, including experiment tracking, model packaging, and deployment.
shap - Installs SHAP (SHapley Additive exPlanations), a library used for "model explainability." it helps you understand how different features in your data contribute to a model's specific predictions.
-q -  A flag that tells pip to be "quiet." It hides non-error messages (like progress bars and successful download notifications) to keep your terminal or notebook output clean.
'''

'\nmlflow - Installs MLflow, an open-source platform used to manage the machine learning lifecycle, including experiment tracking, model packaging, and deployment.\nshap - Installs SHAP (SHapley Additive exPlanations), a library used for "model explainability." it helps you understand how different features in your data contribute to a model\'s specific predictions.\n-q -  A flag that tells pip to be "quiet." It hides non-error messages (like progress bars and successful download notifications) to keep your terminal or notebook output clean.\n'

In [8]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')


In [9]:
data  = load_diabetes()
X     = pd.DataFrame(data.data, columns=data.feature_names)
y     = pd.Series(data.target, name='disease_progression')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Any nulls: {X.isnull().sum().sum()}")

Train: (353, 10) | Test: (89, 10)
Any nulls: 0


In [10]:
# Imputer + Scaler + Model in one object
# Preprocessing fits on TRAINING data only — no data leakage
pipeline_v1 = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # step 1: fill nulls
    ('scaler',  StandardScaler()),                  # step 2: normalize
    ('model',   RandomForestRegressor(              # step 3: train
                    n_estimators=100,
                    max_depth=5,
                    random_state=42
                ))
])

pipeline_v1.fit(X_train, y_train)
preds = pipeline_v1.predict(X_test)
r2    = r2_score(y_test, preds)
mae   = mean_absolute_error(y_test, preds)

print(f"Pipeline v1 — Imputer + Scaler + Random Forest")
print(f"R²  : {r2:.3f}")
print(f"MAE : {mae:.1f}")
print(f"Steps: {[s[0] for s in pipeline_v1.steps]}")

Pipeline v1 — Imputer + Scaler + Random Forest
R²  : 0.455
MAE : 43.6
Steps: ['imputer', 'scaler', 'model']


In [11]:
# In real healthcare data you have mixed feature types
# ColumnTransformer applies different steps to different columns
# Same thinking as your silver layer — different rules per column type

numeric_features = ['age', 'bmi', 'bp', 's1', 's2', 's3']
other_features   = ['sex', 's4', 's5', 's6']

# RobustScaler for numeric — handles outliers better than StandardScaler
# Critical in clinical data where extreme lab values are common
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

other_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_transformer, numeric_features),
    ('other',   other_transformer,   other_features)
])

print("ColumnTransformer built ✓")
print(f"Numeric features: {numeric_features}")
print(f"Other features  : {other_features}")

ColumnTransformer built ✓
Numeric features: ['age', 'bmi', 'bp', 's1', 's2', 's3']
Other features  : ['sex', 's4', 's5', 's6']


In [12]:
pipeline_v2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ))
])

cv_scores = cross_val_score(pipeline_v2, X, y, cv=5, scoring='r2')
pipeline_v2.fit(X_train, y_train)

preds = pipeline_v2.predict(X_test)
r2    = r2_score(y_test, preds)
mae   = mean_absolute_error(y_test, preds)

print("Pipeline v2 — ColumnTransformer + Random Forest")
print(f"R²        : {r2:.3f}")
print(f"MAE       : {mae:.1f}")
print(f"CV Mean R²: {cv_scores.mean():.3f}")
print(f"CV Std    : {cv_scores.std():.3f}")
print(f"\nv1 R²: 0.441 → v2 R²: {r2:.3f}")

Pipeline v2 — ColumnTransformer + Random Forest
R²        : 0.452
MAE       : 43.8
CV Mean R²: 0.430
CV Std    : 0.064

v1 R²: 0.441 → v2 R²: 0.452


In [13]:
!pip install xgboost -q
from xgboost import XGBRegressor

# The power of Pipeline — swap just the model, keep preprocessing unchanged
# Like swapping gold layer logic while keeping bronze and silver the same

models_to_try = {
    'Random Forest'    : RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'Linear Regression': LinearRegression(),
    'XGBoost'          : XGBRegressor(n_estimators=100, max_depth=5, random_state=42, verbosity=0)


        # In software, verbosity controls how much "chatter" or background information the computer prints to your screen while it is working.
        # For XGBoost, the verbosity parameter specifically handles how many warning and information messages you see in your console or notebook:
        # 0 (Silent): Only shows critical errors. This is the cleanest setting and is what your code is using.
        # 1 (Warning): The default. It shows errors and warnings (like if a feature is being deprecated).
        # 2 (Info): Prints basic status updates while the model is training.
        # 3 (Debug): Prints everything—every detail of what the algorithm is doing behind the scenes. This is usually only used if you are trying to fix a bug.

}

mlflow.set_experiment("diabetes-pipeline-comparison")

results = []
for name, mdl in models_to_try.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', mdl)
    ])
    cv  = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    pipe.fit(X_train, y_train)
    r2  = r2_score(y_test, pipe.predict(X_test))
    mae = mean_absolute_error(y_test, pipe.predict(X_test))

    with mlflow.start_run(run_name=f"pipeline-{name.lower().replace(' ','-')}"):
        mlflow.log_param("model",        name)
        mlflow.log_param("pipeline_v",   "v2-columntransformer")
        mlflow.log_metric("r2_test",     round(r2, 3))
        mlflow.log_metric("mae",         round(mae, 1))
        mlflow.log_metric("cv_mean_r2",  round(cv.mean(), 3))
        mlflow.sklearn.log_model(pipe, name.lower().replace(' ','-'))

    results.append({'Model': name, 'CV R²': round(cv.mean(),3),
                    'Test R²': round(r2,3), 'MAE': round(mae,1)})
    print(f"✓ {name:20s} | CV R²: {cv.mean():.3f} | MAE: {mae:.1f}")

df = pd.DataFrame(results).sort_values('CV R²', ascending=False)
print("\nFinal comparison:")
print(df.to_string(index=False))

2026/03/14 21:05:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 21:05:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Random Forest        | CV R²: 0.430 | MAE: 43.8


2026/03/14 21:05:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 21:05:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ Linear Regression    | CV R²: 0.482 | MAE: 42.8


2026/03/14 21:06:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/14 21:06:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ XGBoost              | CV R²: 0.283 | MAE: 45.1

Final comparison:
            Model  CV R²  Test R²  MAE
Linear Regression  0.482    0.453 42.8
    Random Forest  0.430    0.452 43.8
          XGBoost  0.283    0.380 45.1


In [14]:
import joblib

# Save the best pipeline — standard way to save sklearn pipelines
joblib.dump(pipeline_v2, 'diabetes_pipeline_v2.pkl')
print("Pipeline saved ✓")

# Load and test on one patient — simulates a real prediction API call
loaded_pipeline = joblib.load('diabetes_pipeline_v2.pkl')
sample_patient  = X_test.iloc[[0]]
prediction      = loaded_pipeline.predict(sample_patient)
actual          = y_test.iloc[0]

print(f"\nSample prediction test:")
print(f"Predicted : {prediction[0]:.1f}")
print(f"Actual    : {actual:.1f}")
print(f"Difference: {abs(prediction[0]-actual):.1f} points")

# Push to GitHub
!git add .
!git commit -m "feat: day 3 - sklearn pipeline with columntransformer + xgboost"
!git push origin main
print("\nPushed ✓ — Wednesday done!")

Pipeline saved ✓

Sample prediction test:
Predicted : 155.2
Actual    : 219.0
Difference: 63.8 points
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git

Pushed ✓ — Wednesday done!


XGBoost underperformed badly — 0.283 CV R²
This looks shocking but the reason is simple — XGBoost is extremely sensitive to hyperparameters on small datasets. With only 442 rows, default XGBoost overfits hard.

In [15]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline

# Default XGBoost — what you just ran
xgb_default = XGBRegressor(n_estimators=100, max_depth=5,
                             random_state=42, verbosity=0)

# XGBoost tuned for small datasets
xgb_tuned = XGBRegressor(
    n_estimators=100,
    max_depth=3,        # shallower — less overfitting
    learning_rate=0.1,  # slower learning — more stable
    subsample=0.8,      # use 80% of rows per tree
    random_state=42,
    verbosity=0
)

for name, mdl in [('XGBoost default', xgb_default),
                   ('XGBoost tuned',   xgb_tuned)]:
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', mdl)
    ])
    cv = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    print(f"{name:20s} | CV R²: {cv.mean():.3f} | Std: {cv.std():.3f}")

XGBoost default      | CV R²: 0.283 | Std: 0.063
XGBoost tuned        | CV R²: 0.416 | Std: 0.066
